## Group 1: merge and biolink format all tsv files
1. edge types

2. the corresponding config.json file is: config_

3. all of them have same cols "subject, predicate, object, agent_type, knowledge_level, knowledge_source, object_category, publications, subject_category"

4. Full list of tsv files handled in this group is:


In [11]:
## Load necessary packages
import os
import pandas as pd
import glob
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

## Define the version number
version_number = "08_10_2026"
deployment_date = "2026-08-10"

### Load the Biolink category and predicate dictionary for mapping subject, object, and predicate types

In [95]:
## Load the Biolink category and predicate dictionary for mapping subject, object, and predicate types
%run ./Biolink_category_and_predication_dictionary.ipynb

Date of last update:  2026-08-26
Order is to always process Node/category map first, since the Edeg/predicate map depends on biolink-complainat node values
-----------------------------------------------------------------------------------------------------------------------------
Dictionary: category_map, Key template: Subject_category or Object_category
------------------------------------------------------------------------------------------
Dictionary: predicate_map, Key template: (Subject_category, Object_category, Predicate)


### Get all helper functions

In [55]:
## Load all helper functions
%run /Users/Weiqi0/ISB_working/Hadlock_lab/QI_ISB_Git_repo/TranslatorPharcogenomicsKG/Parser_helper_functions.ipynb

## Load files and convert them into separate node & edge files
* check all imported file structure

In [13]:
## Notice!! Please change the file path of following codes into your own
raw_files_path = '/Users/Weiqi0/ISB_working/Ilya_lab/Translator/Pharmagenomics_KG/files/CIViC/'

## Define the output path for node & edge files after formatting
download_path_node_file = f'/Users/Weiqi0/ISB_working/Ilya_lab/Translator/Pharmagenomics_KG/files/parsed/CIViC_parsed_node_{version_number}.tsv'
download_path_edge_file = f'/Users/Weiqi0/ISB_working/Ilya_lab/Translator/Pharmagenomics_KG/files/parsed/CIViC_parsed_edge_{version_number}.tsv'

In [14]:
## Check all node files being read
## Read all BigGIM node csv file in group 1

for f in os.listdir(raw_files_path):
    if f.endswith('.tsv'):
        print(f)

01-Aug-2026-AcceptedAssertionSummaries.tsv
01-Aug-2026-VariantSummaries.tsv
01-Aug-2026-FeatureSummaries.tsv
01-Aug-2026-AcceptedClinicalEvidenceSummaries.tsv
01-Aug-2026-VariantGroupSummaries.tsv
01-Aug-2026-MolecularProfileSummaries.tsv


### Edge & node files & edge design
* edge: AcceptedClinicalEvidenceSummaries
* node: VariantSummaries
    * subject:   molecular_profile  (biolink:SequenceVariant — same CAID/ClinGen resolution issue as before)
    * predicate: <depends on evidence_type + significance + evidence_direction>
    * object:    depends on evidence_type:
         - Diagnostic    -> disease (biolink:Disease)
         - Predictive    -> therapies (biolink:Drug)
         - Prognostic    -> disease/phenotypes (biolink:Disease / biolink:PhenotypicFeature)
         - Predisposing  -> disease (biolink:Disease)
         - Oncogenic     -> disease (biolink:Disease)
         - Functional    -> gene (biolink:Gene)
    * Examples:
        * "id": "civic.eid:1",
        * "subject": "<CAID or civic.mp:64 for JAK2 V617F>",
        * "subject_category": "biolink:SequenceVariant",
      * "predicate": "biolink:contributes_to",
      * "object": "DOID:1037",
      * "object_category": "biolink:Disease",
      * "negated": true,
      * "qualifiers": {
        * "context_qualifier": "diagnostic"
      * },
      * "attributes":

In [26]:
## Read each potential tsv files for nodes
variant_node_df = pd.read_csv(raw_files_path + '01-Aug-2026-VariantSummaries.tsv', sep='\t')
variant_node_df.head(2)

,variant_id,variant_civic_url,feature_type,feature_id,feature_name,feature_civic_url,variant,variant_aliases,is_flagged,variant_groups,...,vicc_compliant_name,5_prime_transcript,5_prime_end_exon,5_prime_exon_offset,5_prime_exon_offset_direction,3_prime_transcript,3_prime_start_exon,3_prime_exon_offset,3_prime_exon_offset_direction,iscn_name
0,1,https://civicdb.org/links/variants/1,Fusion,61802,BCR::ABL1,https://civicdb.org/links/features/61802,Fusion,"T(9;22)(Q34;Q11),BCR-ABL1,BCR-ABL",False,ABL1 fusions in B-ALL,...,BCR(entrez:613)::ABL1(entrez:25),ENST00000305877.8,14.0,NaN,NaN,ENST00000318560.5,2.0,NaN,NaN,NaN
1,2,https://civicdb.org/links/variants/2,Gene,4,ABL1,https://civicdb.org/links/features/4,T315I,"THR334ILE,RS121913459",False,Imatinib Resistance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [45]:
print(variant_node_df.columns)

Index(['variant_id', 'variant_civic_url', 'feature_type', 'feature_id',
       'feature_name', 'feature_civic_url', 'variant', 'variant_aliases',
       'is_flagged', 'variant_groups', 'variant_types',
       'single_variant_molecular_profile_id', 'last_review_date', 'gene',
       'entrez_id', 'chromosome', 'start', 'stop', 'reference_bases',
       'variant_bases', 'representative_transcript', 'ensembl_version',
       'reference_build', 'hgvs_descriptions', 'allele_registry_id',
       'clinvar_ids', 'ncit_id', '5_prime_partner_status', '5_prime_partner',
       '3_prime_partner_status', '3_prime_partner', 'vicc_compliant_name',
       '5_prime_transcript', '5_prime_end_exon', '5_prime_exon_offset',
       '5_prime_exon_offset_direction', '3_prime_transcript',
       '3_prime_start_exon', '3_prime_exon_offset',
       '3_prime_exon_offset_direction', 'iscn_name'],
      dtype='object')


In [27]:
## Read each potential tsv files for nodes
edge_df = pd.read_csv(raw_files_path + '01-Aug-2026-AcceptedClinicalEvidenceSummaries.tsv', sep='\t')

edge_df.head(2)

,molecular_profile,molecular_profile_id,disease,doid,phenotypes,therapies,therapy_interaction_type,evidence_type,evidence_direction,evidence_level,...,citation,nct_ids,rating,evidence_status,evidence_id,variant_origin,last_review_date,evidence_civic_url,molecular_profile_civic_url,is_flagged
0,JAK2 V617F,64,Lymphoid Leukemia,1037.0,NaN,NaN,NaN,Diagnostic,Supports,B,...,"Levine et al., 2005",NaN,4.0,accepted,1,Somatic,2023-01-09 21:46:26 UTC,https://civicdb.org/links/evidence_items/1,https://civicdb.org/links/molecular_profiles/64,False
1,PDGFRA D842V,99,Gastrointestinal Stromal Tumor,9253.0,NaN,NaN,NaN,Diagnostic,Supports,B,...,"Lasota et al., 2004",NaN,3.0,accepted,2,Somatic,2023-01-09 21:46:27 UTC,https://civicdb.org/links/evidence_items/2,https://civicdb.org/links/molecular_profiles/99,False


In [28]:
print(len(edge_df))

4878


In [29]:
print(edge_df.columns)

Index(['molecular_profile', 'molecular_profile_id', 'disease', 'doid',
       'phenotypes', 'therapies', 'therapy_interaction_type', 'evidence_type',
       'evidence_direction', 'evidence_level', 'significance',
       'evidence_statement', 'citation_id', 'source_type', 'asco_abstract_id',
       'citation', 'nct_ids', 'rating', 'evidence_status', 'evidence_id',
       'variant_origin', 'last_review_date', 'evidence_civic_url',
       'molecular_profile_civic_url', 'is_flagged'],
      dtype='object')


In [31]:
## Filtering and see there are how many drug-gene pairs in each categories of column "PGx on FDA Label"
# Count of each distinct value
print(edge_df[['evidence_type', 'significance', 'evidence_direction']].value_counts())

evidence_type  significance            evidence_direction
Predictive     Sensitivity/Response    Supports              1617
               Resistance              Supports               827
Predisposing   Predisposition          Supports               534
Diagnostic     Positive                Supports               483
Prognostic     Poor Outcome            Supports               343
Predictive     Resistance              Does Not Support       218
               Sensitivity/Response    Does Not Support       166
Oncogenic      Oncogenicity            Supports               134
Predisposing   Uncertain Significance  Supports               127
Prognostic     Better Outcome          Supports               118
Functional     Gain of Function        Supports                43
               Dominant Negative       Supports                41
                                       Does Not Support        40
Prognostic     Poor Outcome            Does Not Support        30
Functional     Los

### Now filter out those low quality combos
* all those edges with col('evidence_direction') == "Does Not Support"

In [32]:
## Filter out rows with null, no Clinical PGx values in col('PGx on FDA Label')
print("Before filter, row counts: ", len(edge_df))

edge_filter_df = edge_df.dropna(subset=['evidence_direction'])
edge_filter_df = edge_filter_df[~edge_filter_df['evidence_direction'].isin(['Does Not Support'])]

print("After filter, row counts: ", len(edge_filter_df))

Before filter, row counts:  4878
After filter, row counts:  4343


In [33]:
## Filtering and see there are how many drug-gene pairs in each categories of column "PGx on FDA Label"
# Count of each distinct value
print(edge_filter_df[['evidence_type', 'significance', 'evidence_direction']].value_counts())

evidence_type  significance            evidence_direction
Predictive     Sensitivity/Response    Supports              1617
               Resistance              Supports               827
Predisposing   Predisposition          Supports               534
Diagnostic     Positive                Supports               483
Prognostic     Poor Outcome            Supports               343
Oncogenic      Oncogenicity            Supports               134
Predisposing   Uncertain Significance  Supports               127
Prognostic     Better Outcome          Supports               118
Functional     Gain of Function        Supports                43
               Dominant Negative       Supports                41
               Loss of Function        Supports                26
Predictive     Reduced Sensitivity     Supports                16
               Adverse Response        Supports                10
Diagnostic     Negative                Supports                 9
Functional     Una

In [40]:
## check rows with col("evidence_type") == Functional
edge_functional_type_df = edge_filter_df[edge_filter_df['evidence_type'].isin(['Functional'])]
print(len(edge_functional_type_df))
edge_functional_type_df.head(2)

118


,molecular_profile,molecular_profile_id,disease,doid,phenotypes,therapies,therapy_interaction_type,evidence_type,evidence_direction,evidence_level,...,citation,nct_ids,rating,evidence_status,evidence_id,variant_origin,last_review_date,evidence_civic_url,molecular_profile_civic_url,is_flagged
3436,TP53 G245S,853,NaN,NaN,NaN,NaN,NaN,Functional,Supports,D,...,"Billant et al., 2016",NaN,3.0,accepted,7114,Unknown,2025-11-26 21:24:17 UTC,https://civicdb.org/links/evidence_items/7114,https://civicdb.org/links/molecular_profiles/853,False
3437,TP53 R248Q,117,NaN,NaN,NaN,NaN,NaN,Functional,Supports,D,...,"Billant et al., 2016",NaN,3.0,accepted,7115,Unknown,2025-11-26 21:35:24 UTC,https://civicdb.org/links/evidence_items/7115,https://civicdb.org/links/molecular_profiles/117,False


### Remove rows with col("evidence_type") == "Functional"
* since those rows are lack of object column

In [41]:
## Filter out rows with null, no Clinical PGx values in col('PGx on FDA Label')
print("Before filter, row counts: ", len(edge_filter_df))

edge_filter_df2 = edge_filter_df.dropna(subset=['evidence_type'])
edge_filter_df2 = edge_filter_df2[~edge_filter_df['evidence_type'].isin(['Functional'])]

print("After filter, row counts: ", len(edge_filter_df2))

Before filter, row counts:  4343
After filter, row counts:  4225


In [42]:
## check rows with col("evidence_type") == Functional
edge_predictive_type_df = edge_filter_df2[edge_filter_df2['evidence_type'].isin(['Predictive'])]
print(len(edge_predictive_type_df))
edge_predictive_type_df.head(2)

2470


,molecular_profile,molecular_profile_id,disease,doid,phenotypes,therapies,therapy_interaction_type,evidence_type,evidence_direction,evidence_level,...,citation,nct_ids,rating,evidence_status,evidence_id,variant_origin,last_review_date,evidence_civic_url,molecular_profile_civic_url,is_flagged
9,MAP2K1 P124S,82,Melanoma,1909.0,NaN,Selumetinib,NaN,Predictive,Supports,D,...,"Emery et al., 2009",NaN,3.0,accepted,12,Somatic,2023-01-09 21:46:27 UTC,https://civicdb.org/links/evidence_items/12,https://civicdb.org/links/molecular_profiles/82,False
10,MAP2K1 Q56P,83,Melanoma,1909.0,NaN,Selumetinib,NaN,Predictive,Supports,D,...,"Emery et al., 2009",NaN,3.0,accepted,13,Somatic,2023-01-09 21:46:27 UTC,https://civicdb.org/links/evidence_items/13,https://civicdb.org/links/molecular_profiles/83,False


In [43]:
## check all unique molecular values
counts = edge_filter_df2['molecular_profile'].value_counts()
print(len(counts))

1720


In [44]:
## check all unique molecular values
counts = edge_filter_df2['disease'].value_counts()
print(len(counts))

331


### Now trying to inner join and use the variant node file to fill identifiers for the edge file
* select cols: feature_type, feature_name, variant, single_variant_molecular_profile_id, representative_transcript, ensembl_version, clinvar_ids

In [46]:
sele_cols = ['feature_type', 'feature_name', 'variant', 'single_variant_molecular_profile_id', 'representative_transcript', 'ensembl_version', 'clinvar_ids']

variant_node_select_df = variant_node_df[sele_cols].drop_duplicates()

In [51]:
## check all unique molecular values
counts = variant_node_select_df['feature_type'].value_counts()
print(counts)

feature_type
Gene      1738
Fusion     241
Factor      13
Name: count, dtype: int64


In [49]:
print("before remove those without a valid representative_transcript identifier: ", len(variant_node_select_df))
variant_node_select_noNaN_df = variant_node_select_df.dropna(subset=['representative_transcript'])
print("after remove those without a valid representative_transcript identifier: ", len(variant_node_select_noNaN_df))                                                                                 

before remove those without a valid representative_transcript identifier:  1992
after remove those without a valid representative_transcript identifier:  1244


In [50]:
## check all unique molecular values
counts = variant_node_select_noNaN_df['feature_type'].value_counts()
print(counts)

feature_type
Gene    1244
Name: count, dtype: int64


In [52]:
variant_node_select_noNaN_df.head(2)

,feature_type,feature_name,variant,single_variant_molecular_profile_id,representative_transcript,ensembl_version,clinvar_ids
1,Gene,ABL1,T315I,2,ENST00000318560.5,75.0,12624
2,Gene,ABL1,E255K,3,ENST00000318560.5,75.0,376090


In [56]:
## now split the column: representative_transcript into two parts, first is ensembl_id and ensembl_id_version
variant_node_splited_df = add_ensembl_curie_columns(variant_node_select_noNaN_df, "representative_transcript")

variant_node_splited_df.head(2)

,feature_type,feature_name,variant,single_variant_molecular_profile_id,representative_transcript,ensembl_version,clinvar_ids,ensembl_identifier,ensembl_id_version
1,Gene,ABL1,T315I,2,ENST00000318560.5,75.0,12624,ENSEMBL:ENST00000318560,5
2,Gene,ABL1,E255K,3,ENST00000318560.5,75.0,376090,ENSEMBL:ENST00000318560,5


In [57]:
## Select only needed columns
sele_cols = ['single_variant_molecular_profile_id', 'ensembl_identifier', 'ensembl_version', 'ensembl_id_version']

variant_node_splited_select_df = variant_node_splited_df[sele_cols].drop_duplicates()

In [63]:
variant_node_splited_select_df.head(2)

,single_variant_molecular_profile_id,ensembl_identifier,ensembl_version,ensembl_id_version
1,2,ENSEMBL:ENST00000318560,75.0,5
2,3,ENSEMBL:ENST00000318560,75.0,5


### Then inner join the pandas variant_node_splited_select_df with edge_filter_df2 using single_variant_molecular_profile_id and molecular_profile_id columns

In [58]:
print(edge_filter_df2.columns)

Index(['molecular_profile', 'molecular_profile_id', 'disease', 'doid',
       'phenotypes', 'therapies', 'therapy_interaction_type', 'evidence_type',
       'evidence_direction', 'evidence_level', 'significance',
       'evidence_statement', 'citation_id', 'source_type', 'asco_abstract_id',
       'citation', 'nct_ids', 'rating', 'evidence_status', 'evidence_id',
       'variant_origin', 'last_review_date', 'evidence_civic_url',
       'molecular_profile_civic_url', 'is_flagged'],
      dtype='object')


In [59]:
## Select only needed columns
sele_cols = ['molecular_profile', 'molecular_profile_id', 'disease', 'doid',
       'therapies', 'evidence_type',
       'evidence_direction', 'evidence_level', 'significance',
       'evidence_statement', 'citation_id', 'source_type', 'asco_abstract_id',
       'citation']

edge_filter_select_df2 = edge_filter_df2[sele_cols].drop_duplicates()

In [60]:
## Filtering and see there are how many drug-gene pairs in each categories of column "PGx on FDA Label"
# Count of each distinct value
print(edge_filter_select_df2[['evidence_type', 'significance', 'evidence_direction']].value_counts())

evidence_type  significance            evidence_direction
Predictive     Sensitivity/Response    Supports              1615
               Resistance              Supports               826
Predisposing   Predisposition          Supports               534
Diagnostic     Positive                Supports               483
Prognostic     Poor Outcome            Supports               343
Oncogenic      Oncogenicity            Supports               134
Predisposing   Uncertain Significance  Supports               126
Prognostic     Better Outcome          Supports               118
Predictive     Reduced Sensitivity     Supports                16
               Adverse Response        Supports                10
Diagnostic     Negative                Supports                 9
Predisposing   Protectiveness          Supports                 1
Name: count, dtype: int64


In [61]:
## Since finding identifiers for drug combinations are tough
## skip all those edges with evidence_type == predictive for now
edge_filter_select_df3 = edge_filter_select_df2[~edge_filter_select_df2['evidence_type'].isin(['Predictive'])]

In [62]:
## Filtering and see there are how many drug-gene pairs in each categories of column "PGx on FDA Label"
# Count of each distinct value
print(edge_filter_select_df3[['evidence_type', 'significance', 'evidence_direction']].value_counts())

evidence_type  significance            evidence_direction
Predisposing   Predisposition          Supports              534
Diagnostic     Positive                Supports              483
Prognostic     Poor Outcome            Supports              343
Oncogenic      Oncogenicity            Supports              134
Predisposing   Uncertain Significance  Supports              126
Prognostic     Better Outcome          Supports              118
Diagnostic     Negative                Supports                9
Predisposing   Protectiveness          Supports                1
Name: count, dtype: int64


In [85]:
edge_merged_df = pd.merge(
    edge_filter_select_df3,
    variant_node_splited_select_df,
    left_on="molecular_profile_id",
    right_on="single_variant_molecular_profile_id",
    how="inner",
).drop_duplicates()

In [86]:
## Filtering and see there are how many drug-gene pairs in each categories of column "PGx on FDA Label"
# Count of each distinct value
print(edge_merged_df[['evidence_type', 'significance', 'evidence_direction']].value_counts())

evidence_type  significance            evidence_direction
Predisposing   Predisposition          Supports              485
Prognostic     Poor Outcome            Supports              313
Predisposing   Uncertain Significance  Supports              114
Prognostic     Better Outcome          Supports               98
Diagnostic     Positive                Supports               89
Oncogenic      Oncogenicity            Supports               76
Diagnostic     Negative                Supports                5
Predisposing   Protectiveness          Supports                1
Name: count, dtype: int64


In [87]:
### Add a negated column and the values is True only when evidence_type, significance, and evidence_direction are Diagnostic	Negative	Supports
edge_merged_df["negated"] = (
    (edge_merged_df["evidence_type"] == "Diagnostic") &
    (edge_merged_df["significance"] == "Negative") &
    (edge_merged_df["evidence_direction"] == "Supports")
)

In [88]:
print(edge_merged_df.columns)

Index(['molecular_profile', 'molecular_profile_id', 'disease', 'doid',
       'therapies', 'evidence_type', 'evidence_direction', 'evidence_level',
       'significance', 'evidence_statement', 'citation_id', 'source_type',
       'asco_abstract_id', 'citation', 'single_variant_molecular_profile_id',
       'ensembl_identifier', 'ensembl_version', 'ensembl_id_version',
       'negated'],
      dtype='object')


In [89]:
edge_merged_df = edge_merged_df.dropna(subset=['doid'])

In [90]:
## add subject category columns
edge_merged_df["subject_category"], edge_merged_df["object_category"] = 'biolink:Transcript', 'biolink:Disease'

## Rename columns
edge_merged_df = edge_merged_df.rename(columns={"molecular_profile": "subject_name", "ensembl_identifier": "subject", "disease": "object_name"})

## Cast to standard 'object' dtype
edge_merged_df['object'] = 'DOID:' + edge_merged_df['doid'].astype(int).astype(str)

edge_merged_df.head(2)

,subject_name,molecular_profile_id,object_name,doid,therapies,evidence_type,evidence_direction,evidence_level,significance,evidence_statement,...,asco_abstract_id,citation,single_variant_molecular_profile_id,subject,ensembl_version,ensembl_id_version,negated,subject_category,object_category,object
0,JAK2 V617F,64,Lymphoid Leukemia,1037.0,NaN,Diagnostic,Supports,B,Negative,JAK2 V617F is not associated with lymphoid leu...,...,NaN,"Levine et al., 2005",64,ENSEMBL:ENST00000381652,75.0,3,True,biolink:Transcript,biolink:Disease,DOID:1037
1,PDGFRA D842V,99,Gastrointestinal Stromal Tumor,9253.0,NaN,Diagnostic,Supports,B,Negative,GIST tumors harboring PDGFRA D842V mutation ar...,...,NaN,"Lasota et al., 2004",99,ENSEMBL:ENST00000257290,75.0,5,True,biolink:Transcript,biolink:Disease,DOID:9253


In [97]:
## drop no longer needed columns from edge df
# print(edge_merged_df.columns.tolist())
drop_cols = ['molecular_profile_id', 'doid', 'source_type', 'asco_abstract_id', 'single_variant_molecular_profile_id',]

edge_clean_df = edge_merged_df.drop(drop_cols, axis=1)

print(edge_clean_df.columns.tolist())

['subject_name', 'object_name', 'therapies', 'evidence_type', 'evidence_direction', 'evidence_level', 'significance', 'evidence_statement', 'citation_id', 'citation', 'subject', 'ensembl_version', 'ensembl_id_version', 'negated', 'subject_category', 'object_category', 'object']


In [99]:
### Now read in the dictionary and get the exact mapping of predicates
## evidence_type  significance

## match only combinations allowed in the pair dictionary
## Apply the mapping and return a Series with theses columns
edge_clean_df[['predicate', 'subject_direction_qualifier', 'object_direction_qualifier', 'object_aspect_qualifier']] = edge_clean_df.apply(
    lambda row: pd.Series(
        predicate_map.get(
            (row['subject_category'], row['object_category'], row['evidence_type'], row['significance']),
            [None, None, None]  # Default if not found
        )
    ),
    axis=1
)


In [100]:
edge_clean_df.head(4)

,subject_name,object_name,therapies,evidence_type,evidence_direction,evidence_level,significance,evidence_statement,citation_id,citation,...,ensembl_version,ensembl_id_version,negated,subject_category,object_category,object,predicate,subject_direction_qualifier,object_direction_qualifier,object_aspect_qualifier
0,JAK2 V617F,Lymphoid Leukemia,NaN,Diagnostic,Supports,B,Negative,JAK2 V617F is not associated with lymphoid leu...,16081687,"Levine et al., 2005",...,75.0,3,True,biolink:Transcript,biolink:Disease,DOID:1037,biolink:contributes_to,None,None,NaN
1,PDGFRA D842V,Gastrointestinal Stromal Tumor,NaN,Diagnostic,Supports,B,Negative,GIST tumors harboring PDGFRA D842V mutation ar...,15146165,"Lasota et al., 2004",...,75.0,5,True,biolink:Transcript,biolink:Disease,DOID:9253,biolink:contributes_to,None,None,NaN
2,DNMT3A R882,Acute Myeloid Leukemia,NaN,Diagnostic,Supports,B,Positive,Young AML patients (<60 years old) with DNMT3A...,22490330,"Ribeiro et al., 2012",...,75.0,3,False,biolink:Transcript,biolink:Disease,DOID:9119,biolink:contributes_to,None,None,NaN
3,JAK2 V617F,Chronic Myeloid Leukemia,NaN,Diagnostic,Supports,B,Positive,JAK2 V617F is associated with myeloid malignan...,16081687,"Levine et al., 2005",...,75.0,3,False,biolink:Transcript,biolink:Disease,DOID:8552,biolink:contributes_to,None,None,NaN


## Now add remaining information columns
* 

In [101]:
## Drop those unmatched rows
## Drop rows where 'name' is NaN, None, or empty string
edge_clean_df = edge_clean_df[~edge_clean_df['predicate'].isna() & (edge_clean_df['predicate'].str.strip() != '')]

## add knowledge source column
edge_clean_df['knowledge_source'] = 'CIVIC'

## add a new knowledge_level column and set value to be 'knowledge_assertion'
edge_clean_df['knowledge_level'] = 'knowledge_assertion'

## add a new agent_type column and set value to be 'manual_agent'
edge_clean_df['agent_type'] = 'text_mining_agent'

## Cast to standard 'object' dtype and add prefix
edge_clean_df['publications'] = 'PMID:' + edge_clean_df['citation_id'].astype(int).astype(str)

## add a deployment date label
edge_clean_df['deploy_date'] = deployment_date

## create a context_qualifier column information from the col('therapies') and fill na if this column is empty
## if all of them are empty, then fill na
edge_clean_df['context_qualifier'] = edge_clean_df['therapies']

In [102]:
### Add resources_id column, checking whether edge is already
column_list = ['subject', 'predicate', 'object', 'context_qualifier', 'deploy_date']
# Apply the function to each row to generate UUIDs
edge_clean_df['id'] = edge_clean_df[column_list].apply(generate_uuid, axis=1)

In [106]:
edge_clean_df['provided_by'] = 'CIVIC'

In [107]:
## check remaining columns
edge_clean_df.columns

Index(['subject_name', 'object_name', 'therapies', 'evidence_type',
       'evidence_direction', 'evidence_level', 'significance',
       'evidence_statement', 'citation_id', 'citation', 'subject',
       'ensembl_version', 'ensembl_id_version', 'negated', 'subject_category',
       'object_category', 'object', 'predicate', 'subject_direction_qualifier',
       'object_direction_qualifier', 'object_aspect_qualifier',
       'knowledge_source', 'knowledge_level', 'agent_type', 'publications',
       'deploy_date', 'context_qualifier', 'id', 'provided_by'],
      dtype='object')

In [108]:
edge_parsing_df = edge_clean_df.copy()

### Put additional columns into attribute column
import pandas as pd
import json

ATTRIBUTE_CONFIG = {
    "evidence_type": {
        "attribute_type_id": "biolink:has_qualitative_value",
        "value_type_id": "CIVIC:civic_evidence_type",
        "attribute_source": "infores:civic",
    },
    "significance": {
        "attribute_type_id": "biolink:has_qualitative_value",
        "value_type_id": "CIVIC:civic_significance",
        "attribute_source": "infores:civic",
    },
    "evidence_direction": {
        "attribute_type_id": "biolink:has_qualitative_value",
        "value_type_id": "CIVIC:civic_evidence_direction",
        "attribute_source": "infores:civic",
    },
    "evidence_statement": {
        "attribute_type_id": "biolink:has_evidence",
        "value_type_id": "CIVIC:civic_evidence_statement",
        "attribute_source": "infores:civic",
    },
    "ensembl_version": {
        "attribute_type_id": "biolink:has_qualitative_value",
        "value_type_id": "CIVIC:civic_subject_ensembl_version",
        "attribute_source": "infores:civic",
    },
    "ensembl_id_version": {
        "attribute_type_id": "biolink:has_qualitative_value",
        "value_type_id": "CIVIC:civic_subject_ensembl_id_version",
        "attribute_source": "infores:civic",
    },
}

def group_into_attributes(row, config):
    return [
        {
            "attribute_type_id": meta["attribute_type_id"],
            "value": str(row[col]).strip(),
            "value_type_id": meta["value_type_id"],
            "original_attribute_name": col,
            "attribute_source": meta["attribute_source"],
        }
        for col, meta in config.items()
        if pd.notna(row[col]) and str(row[col]).strip() != ""
    ]

edge_parsing_df["attributes"] = edge_parsing_df.apply(lambda r: group_into_attributes(r, ATTRIBUTE_CONFIG), axis=1)

In [109]:
## reorder the dataframe
desired_order = [
    'subject', 'subject_name', 'subject_category', 'object', 'object_name', 'object_category', 
    'predicate', 'negated', 'subject_direction_qualifier', 'object_direction_qualifier', 'object_aspect_qualifier',
    'provided_by', 'knowledge_source', 'publications', 'knowledge_level', 'context_qualifier', 'agent_type', 
    'deploy_date', 'id', 'attributes'
]

edge_parsing_df = edge_parsing_df[desired_order]

print(edge_parsing_df.columns.tolist())

['subject', 'subject_name', 'subject_category', 'object', 'object_name', 'object_category', 'predicate', 'negated', 'subject_direction_qualifier', 'object_direction_qualifier', 'object_aspect_qualifier', 'provided_by', 'knowledge_source', 'publications', 'knowledge_level', 'context_qualifier', 'agent_type', 'deploy_date', 'id', 'attributes']


### now create the corresponding node dataframe directly from the finalized edge dataframe
* only need three columns: id, name, category

In [110]:
node_subject_df = edge_parsing_df[['subject', 'subject_name', 'subject_category']]
node_object_df = edge_parsing_df[['object', 'object_name', 'object_category']]

## rename those columns into desired format
node_subject_df.rename(columns={'subject': 'id', 'subject_name': 'name', 'subject_category': 'category'}, inplace=True)
node_object_df.rename(columns={'object': 'id', 'object_name': 'name', 'object_category': 'category'}, inplace=True)

concat_node_df = pd.concat([node_subject_df, node_object_df]).drop_duplicates(keep='first')

concat_node_df.head(2)

/var/folders/ml/cjwyk2ps62361rr96_3b_n080000gp/T/ipykernel_46447/1675134899.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  node_subject_df.rename(columns={'subject': 'id', 'subject_name': 'name', 'subject_category': 'category'}, inplace=True)
/var/folders/ml/cjwyk2ps62361rr96_3b_n080000gp/T/ipykernel_46447/1675134899.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  node_object_df.rename(columns={'object': 'id', 'object_name': 'name', 'object_category': 'category'}, inplace=True)


,id,name,category
0,ENSEMBL:ENST00000381652,JAK2 V617F,biolink:Transcript
1,ENSEMBL:ENST00000257290,PDGFRA D842V,biolink:Transcript


## following codes are used for quality control and sanity check
* check and confirm all subject & object types are correctly formatted

In [111]:
## check all unique predicate values
counts = concat_node_df['category'].value_counts()
print(counts)

category
biolink:Transcript    569
biolink:Disease       129
Name: count, dtype: int64


In [112]:
## check all unique predicate values
counts = edge_parsing_df['predicate'].value_counts()
print(counts)

predicate
biolink:contributes_to     653
biolink:associated_with    411
biolink:related_to         114
Name: count, dtype: int64


In [113]:
## Create a graph from the DataFrame
graph = nx.from_pandas_edgelist(edge_parsing_df, 'subject', 'object', edge_attr='predicate')

## Print graph information
print('Number of nodes', len(set(graph.nodes)))
print('Number of edges', len(set(graph.edges)))
print('Average degree', sum(dict(graph.degree).values()) / len(graph.nodes))

Number of nodes 280
Number of edges 319
Average degree 2.2785714285714285


## Now output those parsed files
*

In [114]:
## Define the output path for node & edge files after formatting
download_path_node_file = f'/Users/Weiqi0/ISB_working/Ilya_lab/Translator/Pharmagenomics_KG/files/parsed/CIViC_parsed_node_{version_number}.tsv'
download_path_edge_file = f'/Users/Weiqi0/ISB_working/Ilya_lab/Translator/Pharmagenomics_KG/files/parsed/CIViC_parsed_edge_{version_number}.tsv'

## download both node and edge files
## Download the result df
## disable download for testing
concat_node_df.to_csv(download_path_node_file, sep ='\t', index=False)
edge_parsing_df.to_csv(download_path_edge_file, sep ='\t', index=False)